In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter

In [2]:
# --- Phase 1: Load data ---
expr = pd.read_csv('../data/raw/HiSeqV2', sep='\t', index_col=0)
mut = pd.read_csv('../data/raw/BRCA_mc3_gene_level.txt', sep='\t', index_col=0)
clin = pd.read_csv('../data/raw/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix', sep='\t', index_col=0)

In [3]:
# --- Phase 2: Align ---
expr_samples = set(expr.columns)
mut_samples = set(mut.columns)
labeled_patients = set(clin[clin['PAM50Call_RNAseq'].notna()].index)
common = expr_samples & mut_samples & labeled_patients
common_patients = sorted(common)

expr_subset = expr[common_patients].T
mut_subset = mut[common_patients].T
labels = clin.loc[common_patients, 'PAM50Call_RNAseq']

In [4]:

# --- Phase 3: Feature filtering ---
gene_variances = expr_subset.var(axis=0)
top_genes = gene_variances.sort_values(ascending=False).head(2000).index
missing_markers = ['ERBB2', 'MKI67']
final_genes = list(top_genes) + missing_markers
expr_filtered = expr_subset[final_genes]

min_patients = int(0.01 * len(mut_subset))
mut_filtered = mut_subset.loc[:, (mut_subset.sum(axis=0) >= min_patients)]

le = LabelEncoder()
y = le.fit_transform(labels)

scaler = StandardScaler()
expr_scaled = scaler.fit_transform(expr_filtered)

X_expr_train, X_expr_temp, X_mut_train, X_mut_temp, y_train, y_temp = train_test_split(
    expr_scaled, mut_filtered.values, y, test_size=0.3, stratify=y, random_state=42
)
X_expr_val, X_expr_test, X_mut_val, X_mut_test, y_val, y_test = train_test_split(
    X_expr_temp, X_mut_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_expr_train.shape, X_mut_train.shape, y_train.shape)
print("Val:", X_expr_val.shape, X_mut_val.shape, y_val.shape)
print("Test:", X_expr_test.shape, X_mut_test.shape, y_test.shape)

Train: (415, 2002) (415, 2714) (415,)
Val: (89, 2002) (89, 2714) (89,)
Test: (89, 2002) (89, 2714) (89,)


In [5]:
# --- Phase 5: Autoencoder fusion model ---
X_expr_train_t = torch.tensor(X_expr_train, dtype=torch.float32)
X_mut_train_t = torch.tensor(X_mut_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_expr_val_t = torch.tensor(X_expr_val, dtype=torch.float32)
X_mut_val_t = torch.tensor(X_mut_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

X_expr_test_t = torch.tensor(X_expr_test, dtype=torch.float32)
X_mut_test_t = torch.tensor(X_mut_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_expr_train_t, X_mut_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class MultiOmicsAutoencoder(nn.Module):
    def __init__(self, expr_dim, mut_dim, latent_dim=32, n_classes=5):
        super().__init__()
        self.expr_encoder = nn.Sequential(
            nn.Linear(expr_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, latent_dim)
        )
        self.expr_decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, expr_dim)
        )
        self.mut_encoder = nn.Sequential(
            nn.Linear(mut_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, latent_dim)
        )
        self.mut_decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, mut_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim * 2, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, n_classes)
        )

    def forward(self, expr, mut):
        latent_expr = self.expr_encoder(expr)
        latent_mut = self.mut_encoder(mut)
        recon_expr = self.expr_decoder(latent_expr)
        recon_mut = self.mut_decoder(latent_mut)
        fused = torch.cat([latent_expr, latent_mut], dim=1)
        logits = self.classifier(fused)
        return logits, recon_expr, recon_mut
torch.manual_seed(42)
model = MultiOmicsAutoencoder(expr_dim=X_expr_train.shape[1], mut_dim=X_mut_train.shape[1])
optimizer = optim.Adam(model.parameters(), lr=1e-3)

class_counts = Counter(y_train)
weights = torch.tensor([1.0 / class_counts.get(i, 1) for i in range(5)], dtype=torch.float32)
weights = weights / weights.sum() * 5

criterion_class = nn.CrossEntropyLoss(weight=weights)
criterion_recon = nn.MSELoss()
recon_weight = 0.5

n_epochs = 100
# Modified training loop with early stopping (best val_acc checkpoint)
best_val_acc = 0
best_state = None

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for expr_batch, mut_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits, recon_expr, recon_mut = model(expr_batch, mut_batch)
        loss_class = criterion_class(logits, y_batch)
        loss_recon = criterion_recon(recon_expr, expr_batch) + criterion_recon(recon_mut, mut_batch)
        loss = loss_class + recon_weight * loss_recon
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_logits, _, _ = model(X_expr_val_t, X_mut_val_t)
        val_preds = val_logits.argmax(dim=1)
        val_acc = (val_preds == y_val_t).float().mean().item()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: train_loss={total_loss:.4f}, val_acc={val_acc:.4f}, best_val_acc={best_val_acc:.4f}")

# Load best checkpoint and evaluate on test
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    test_logits, _, _ = model(X_expr_test_t, X_mut_test_t)
    test_preds = test_logits.argmax(dim=1)

print(classification_report(y_test, test_preds.numpy(), target_names=le.classes_))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_preds.numpy()))

Epoch 10: train_loss=4.6874, val_acc=0.8539, best_val_acc=0.8764
Epoch 20: train_loss=3.7434, val_acc=0.8539, best_val_acc=0.8876
Epoch 30: train_loss=3.5095, val_acc=0.8539, best_val_acc=0.8876
Epoch 40: train_loss=3.3389, val_acc=0.8315, best_val_acc=0.8876
Epoch 50: train_loss=2.9616, val_acc=0.8315, best_val_acc=0.8876
Epoch 60: train_loss=2.6939, val_acc=0.8202, best_val_acc=0.8876
Epoch 70: train_loss=2.6432, val_acc=0.8427, best_val_acc=0.8876
Epoch 80: train_loss=2.5544, val_acc=0.8315, best_val_acc=0.8876
Epoch 90: train_loss=2.3886, val_acc=0.8090, best_val_acc=0.8876
Epoch 100: train_loss=2.5594, val_acc=0.7978, best_val_acc=0.8876
              precision    recall  f1-score   support

       Basal       0.94      0.94      0.94        16
        Her2       1.00      0.57      0.73         7
        LumA       0.86      0.96      0.91        45
        LumB       0.82      0.78      0.80        18
      Normal       0.00      0.00      0.00         3

    accuracy           

In [6]:
# Ablation: zero out one modality's latent and see how much accuracy drops

model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    # Full model (both modalities) - baseline
    logits_full, _, _ = model(X_expr_test_t, X_mut_test_t)
    preds_full = logits_full.argmax(dim=1)
    acc_full = (preds_full == y_test_t).float().mean().item()

    # Expression only - zero out mutation input
    zero_mut = torch.zeros_like(X_mut_test_t)
    logits_expr_only, _, _ = model(X_expr_test_t, zero_mut)
    preds_expr_only = logits_expr_only.argmax(dim=1)
    acc_expr_only = (preds_expr_only == y_test_t).float().mean().item()

    # Mutation only - zero out expression input
    zero_expr = torch.zeros_like(X_expr_test_t)
    logits_mut_only, _, _ = model(zero_expr, X_mut_test_t)
    preds_mut_only = logits_mut_only.argmax(dim=1)
    acc_mut_only = (preds_mut_only == y_test_t).float().mean().item()

print(f"Full model (both modalities):  {acc_full:.4f}")
print(f"Expression only (mutation zeroed): {acc_expr_only:.4f}")
print(f"Mutation only (expression zeroed): {acc_mut_only:.4f}")

Full model (both modalities):  0.8539
Expression only (mutation zeroed): 0.8539
Mutation only (expression zeroed): 0.4494


In [7]:
import pickle, os
import numpy as np

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../results/models', exist_ok=True)

X_test_concat = np.hstack([X_expr_test, X_mut_test])

np.savez('../data/processed/test_data.npz',
         X_expr_test=X_expr_test, X_mut_test=X_mut_test, y_test=y_test, X_test_concat=X_test_concat)

meta = {
    'expr_features': list(expr_filtered.columns),
    'mut_features': list(mut_filtered.columns),
    'label_classes': le.classes_,
    'cm_autoencoder': confusion_matrix(y_test, test_preds.numpy()),
    'acc_full': acc_full,
    'acc_expr_only': acc_expr_only,
    'acc_mut_only': acc_mut_only,
}

with open('../data/processed/metadata.pkl', 'wb') as f:
    pickle.dump(meta, f)

torch.save(best_state, '../results/models/autoencoder_state.pt')
torch.save({'expr_dim': X_expr_train.shape[1], 'mut_dim': X_mut_train.shape[1]},
           '../results/models/autoencoder_config.pt')

print("Done. Files saved:")
print(os.listdir('../data/processed'))
print(os.listdir('../results/models'))

Done. Files saved:
['expression_clean.csv', 'labels.csv', 'metadata.pkl', 'mutation_clean.csv', 'test_data.npz']
['autoencoder_config.pt', 'autoencoder_state.pt', 'xgboost_model.pkl']
